# RO change over time

## Imports

In [ ]:
import datetime
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import src.XRO
import copy
import scipy.stats
import warnings
import calendar

# import gsw

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Functions

In [ ]:
def get_rolling_std(data, n=20):
    """
    Get standard deviation, computing over time and ensemble member. To increase
    sample size for variance estimate, compute over time window of 2n+1
    years, centered at given year.
    """

    ## do the computation
    kwargs = dict(fn=np.std, n=n, reduce_ensemble_dim=False)
    data_std = src.utils.get_rolling_fn_bymonth(data, **kwargs)

    ## unstack year and month
    data_std = src.utils.unstack_month_and_year(data_std)

    return data_std


def get_stats(x):
    """helper function to compute plotting bounds for experiment"""
    stats = x.quantile(q=[0.1, 0.5, 0.9], dim="member")
    return stats.rename({"quantile": "q"})


def get_fits_over_time(data_rolling, model, by_member=False, **fit_kwargs):
    """Get RO fits for each ensemble member as a function of time."""

    ## empty list to hold results
    fits = []

    ## loop through years
    for y in tqdm.tqdm(data_rolling.year):

        ## get data for year
        data_y = data_rolling.sel(year=y)

        if by_member:

            ## separate fit for each ensemble member
            fits_ = []
            for m in data_rolling.member:
                with warnings.catch_warnings(action="ignore"):
                    fits_.append(model.fit_matrix(data_y.sel(member=m), **fit_kwargs))
            fit = xr.concat(fits_, dim=data_y.member)

        else:

            ## fit for all ensemble members together
            with warnings.catch_warnings(action="ignore"):
                fit = model.fit_matrix(data_y, **fit_kwargs)

        ## track fits
        fits.append(fit.drop_vars(["time", "X", "Y", "Yfit"]))

    ## put back in xarray
    fits = xr.concat(fits, dim=data_rolling.year)

    return fits


def get_fits_over_time_wrapper(
    data_rolling, model, by_member=False, fname=None, **fit_kwargs
):
    """wrapper function to handle saving/loading"""

    ## function to compute fits
    get_fits = lambda: get_fits_over_time(
        data_rolling, model=model, by_member=by_member, **fit_kwargs
    )

    ## if fname not specified, compute without loading/saving
    if fname is None:
        fits = get_fits()

    else:

        ## full path to file
        fp = pathlib.Path(os.environ["SAVE_FP"], "fits_cesm", fname)

        ## load if it already exists
        if fp.is_file():
            fits = xr.open_dataset(fp)

        ## otherwise, compute and save
        else:
            fits = get_fits()
            fits.to_netcdf(fp)

    return fits


def save(fig, fname, dpi=300):
    """save figure to file"""

    ## get save directory
    save_dir = pathlib.Path(os.environ["SAVE_FP"], "ch3-outline")

    ## get fname
    fname = save_dir / f"{fname}.pdf"

    # fig.savefig(fname, dpi=dpi)
    return


def get_sims_over_time(model, params, **simulation_kwargs):
    """Compute stats over time"""

    ## take ensemble mean if necessary
    if "member" in params.dims:
        params = params.mean("member")

    ## empty list to hold result
    sims = []

    ## loop through years
    for y in tqdm.tqdm(params.year):

        ## do simulation
        sim_y = model.simulate(fit_ds=params.sel(year=y), **simulation_kwargs)

        ## append
        sims.append(sim_y)

    ## put back in xarray
    sims = xr.concat(sims, dim=params.year)

    return sims

## Load data

### T, h

In [ ]:
## open data
Th = src.utils.load_cesm_indices(load_z20=True, load_h_cust=True, max_grad=True)

#### Merged Niño index

In [ ]:
## function to get merged nino3/34 regions
def get_nino_merged(x):
    idx = dict(latitude=slice(-5, 5), longitude=slice(190, 270))
    return x.sel(idx).mean(["longitude", "latitude"])


## spatial data
forced, anom = src.utils.load_consolidated()

## compute
Th["T_m"] = src.utils.reconstruct_wrapper(anom[["sst", "sst_comp"]], get_nino_merged)[
    "sst"
]

#### OHC

In [ ]:
def load_consolidated_wide():
    """utility function to load consolidated data"""

    ## directory with data
    CONS_DIR = pathlib.Path(os.environ["DATA_FP"], "cesm", "consolidated_05")

    ## function to align and open
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    align_and_open = lambda fp: src.utils.align_pop_times(xr.open_dataset(fp), **kwargs)

    ## open data and align pop times
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    forced = align_and_open(CONS_DIR / "forced.nc")
    anom = align_and_open(CONS_DIR / "anom.nc")

    return forced, anom


forced_wide, anom_wide = load_consolidated_wide()

In [ ]:
lon_avg = lambda x, lons: x.sel(longitude=slice(*lons)).mean("longitude")

## compute ohc
Th["h_w_ohc"] = src.utils.reconstruct_wrapper(
    anom_wide[["T", "T_comp"]],
    lambda x: lon_avg(x.integrate("z_t"), (120, 180)),
)["T"]

## compute ohc
Th["h_ohc"] = src.utils.reconstruct_wrapper(
    anom_wide[["T", "T_comp"]],
    lambda x: lon_avg(x.integrate("z_t"), (120, 280)),
)["T"]

### NHF

In [ ]:
## get NHF in Niño regions
nhf_3 = src.utils.reconstruct_wrapper(
    anom[["nhf", "nhf_comp"]], fn=src.utils.get_nino3
)["nhf"]
nhf_34 = src.utils.reconstruct_wrapper(
    anom[["nhf", "nhf_comp"]], fn=src.utils.get_nino34
)["nhf"]
nhf_m = src.utils.reconstruct_wrapper(anom[["nhf", "nhf_comp"]], fn=get_nino_merged)[
    "nhf"
]
T_m = src.utils.reconstruct_wrapper(anom[["sst", "sst_comp"]], fn=get_nino_merged)[
    "sst"
]

## convert to units of K/mo
sec_per_mo = 8.64e4 * 30
sec_per_yr = 12 * sec_per_mo
rho = 1.02e3
Cp = 4.2e3
H = 50

Q_34 = nhf_34 * sec_per_yr / (rho * Cp * H)
Q_3 = nhf_3 * sec_per_yr / (rho * Cp * H)
Q_m = nhf_m * sec_per_yr / (rho * Cp * H)

## merge with Th data
Th["Q_34"] = Q_34.sel(time=Th.time)
Th["Q_3"] = Q_3.sel(time=Th.time)
Th["Q_m"] = Q_m.sel(time=Th.time)

### preprocess

In [ ]:
## should we remove median?
REMOVE_MEDIAN = True

## standardize (for convenience)
Th /= Th.std()

## get windowed data (used to estimate change in parameters over time)
Th_rolling = src.utils.get_windowed(Th, window_size=480, stride=120)

## compute grad
Th_rolling["dTdx"] = Th_rolling["T_3"] - Th_rolling["T_4"]

## remove median (optional)
if REMOVE_MEDIAN:
    median = Th_rolling.groupby("time.month").median(["time", "member"])
    Th_rolling = Th_rolling.groupby("time.month") - median

## add constant
Th_rolling["ones"] = xr.ones_like(Th_rolling["T_3"])

## Compute time-varying RO parameters

### Fit model

In [ ]:
## get NHF in Niño regions
nhf_3 = src.utils.reconstruct_wrapper(
    anom[["nhf", "nhf_comp"]], fn=src.utils.get_nino3
)["nhf"]
nhf_34 = src.utils.reconstruct_wrapper(
    anom[["nhf", "nhf_comp"]], fn=src.utils.get_nino34
)["nhf"]
nhf_m = src.utils.reconstruct_wrapper(anom[["nhf", "nhf_comp"]], fn=get_nino_merged)[
    "nhf"
]

## convert to units of K/mo
sec_per_mo = 8.64e4 * 30
sec_per_yr = 12 * sec_per_mo
rho = 1.02e3
Cp = 4.2e3
H = 50

Q_34 = nhf_34 * sec_per_yr / (rho * Cp * H)
Q_3 = nhf_3 * sec_per_yr / (rho * Cp * H)
Q_m = nhf_m * sec_per_yr / (rho * Cp * H)

# ## merge with Th data
# Th["Q_34"] = Q_34.sel(time=Th.time)
# Th["Q_3"] = Q_3.sel(time=Th.time)
# Th["Q_m"] = Q_m.sel(time=Th.time)
# Th["T_m"] = T_m.sel(time=Th.time)

Filenames:
- "T_3-h_w_ohc-zero_median_all-members.nc": ```maskb=["h"], maskNT=["T2", "T3", "TH"], maskNH=["T2"]```
- "T_3-h_w_ohc-zero_median.nc": same, but separate fits for each ensemble member
- "T_3-h_w_ohc-zero_median_simple.nc": ```maskNT=["T2", "T3"]```

In [ ]:
## specify variables
varnames, Q_VAR = ["T_3", "h_w_ohc"], "Q_3"
# varnames, Q_VAR = ["T_34", "h_w_ohc"], "Q_34"
# varnames, Q_VAR = ["T_m", "h_w_ohc"], "Q_m"

## parameters for fitting
MODEL = src.XRO.XRO(ncycle=12, ac_order=4, is_forward=True)
fit_kwargs = dict(ac_mask_idx=None, maskNT=["T2", "T3"], maskNH=[])

## get fits
fits = get_fits_over_time_wrapper(
    Th_rolling[varnames],
    model=MODEL,
    by_member=False,
    fname=None,
    # fname=f"{varnames[0]}-{varnames[1]}-zero_median_simple.nc",
    **fit_kwargs,
)

## extract parameters
params = src.utils.get_params(fits=fits, model=MODEL)

## get change from initial period
delta_params = params - params.isel(year=0)

### Extract params

Get coefficients

In [ ]:
## get coefficients
coefs_nl = fits["NROT_Lac"].sel(nro_form=["T2", "T3"])
R = fits["Lac"].isel(ranky=0, rankx=0).expand_dims(nro_form=["T"])
eps = -fits["Lac"].isel(ranky=1, rankx=1)
coefs = xr.concat([R, coefs_nl], dim="nro_form").rename({"nro_form": "form"})

Compute feature matrix for reconstructing nonlinear $R$

In [ ]:
## how should we scale the feature matrix?
## One of {"constant", "by_month", "sliding_by_month"}
SCALE_TYPE = "by_month"

## get scale
sigma_T = Th_rolling["T_3"].groupby("time.month").std(["time", "member"])

## average as needed
if SCALE_TYPE == "constant":
    sigma_T = sigma_T.mean(["year", "month"]) * xr.ones_like(sigma_T)

elif SCALE_TYPE == "by_month":
    sigma_T = sigma_T.mean("year") * xr.ones_like(sigma_T)

## update coordinate names to match
sigma_T = sigma_T.rename({"month": "cycle"}).assign_coords({"cycle": fits["cycle"]})

## scale feature mat by sigma_T
z = np.arange(-3, 3.1, 0.1)
z = xr.DataArray(z, coords=dict(sigma=z))
Z = sigma_T * z.expand_dims({"year": fits.year, "cycle": sigma_T.cycle})

## expand to get coefficients
Z = xr.concat([Z**i for i in range(3)], dim=coefs.form)

## check everything makes sense
print(np.allclose(Z.sel(form="T2").sel(sigma=1, method="nearest"), sigma_T))

Do the reconstruction

In [ ]:
## reconstruct nonlinear R
R_nl = (coefs * Z).sum("form")
R_nl_ann = R_nl.mean("cycle")
eps_ann = eps.mean("cycle")

## Compute time-varying alpha

### Fits

In [ ]:
def psi_(x, a, b, c):
    """base transfer function"""
    return c * np.exp(a * x) + b


def get_psi(x, y):
    """get transfer function from data"""

    ## prep data
    stack = lambda x: x.stack(s=["time", "member"]).values
    x_ = stack(x)
    y_ = stack(y)

    ## get parameter fit
    p, _ = scipy.optimize.curve_fit(
        f=psi_,
        xdata=x_,
        ydata=y_,
        p0=np.array([0.7, 1.7, -1.5]),
        maxfev=2000,
    )

    ## define transfer func
    psi = lambda x: psi_(x, a=p[0], b=p[1], c=p[2])

    return psi


def psi_poly(x, a, b, c, e):
    """base transfer function"""
    return e * x**3 + c * x**2 + b * x + a


def get_psi_poly(x, y):
    """get transfer function from data"""

    ## prep data
    stack = lambda x: x.stack(s=["time", "member"]).values
    x_ = stack(x)
    y_ = stack(y)

    ## get parameter fit
    p, _ = scipy.optimize.curve_fit(
        f=psi_poly,
        xdata=x_,
        ydata=y_,
    )

    ## define transfer func
    psi = lambda x: psi_poly(x, a=p[0], b=p[1], c=p[2], e=p[3])

    return psi


def get_psi_wrapper(xy, x_var, y_var, year, get_psi_func=get_psi):
    """get transfer function from data"""

    ## get transfer func
    psi_func = get_psi_func(x=xy[x_var].sel(year=year), y=xy[y_var].sel(year=year))

    ## get coefficient array (standardized units)
    z = np.arange(-3, 3.1, 0.1)
    z = xr.DataArray(z, coords=dict(sigma=z))

    ## evaluate function
    psi_eval = xr.zeros_like(z)
    psi_eval.values = psi_func(z.values)

    return psi_eval

In [ ]:
## empty array to hold alpha
psi_ot = []

## args for curve fit
kwargs = dict(x_var=varnames[0], y_var=Q_VAR, get_psi_func=get_psi)

## loop thru
for yr in tqdm.tqdm(Th_rolling.year.isel(year=slice(None, -1))):
    psi_ot.append(
        Th_rolling.groupby("time.month").map(get_psi_wrapper, year=yr, **kwargs)
    )

## concat
psi_ot = xr.concat(psi_ot, dim=Th_rolling.year.isel(year=slice(None, -1)))

## compute alpha
alpha = -psi_ot.differentiate("sigma")

## update coords to match R
alpha = alpha.rename({"month": "cycle"})
alpha["cycle"] = R_nl.cycle

Plot for given month, year

In [ ]:
plot_month = 12
plot_year = 2060

plot_data = Th_rolling.sel(year=plot_year).isel(time=slice(plot_month - 1, None, 12))

fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(plot_data["T_m"], plot_data["Q_m"], s=5)
ax.plot(psi_ot.sigma, psi_ot.sel(month=plot_month, year=plot_year), c="k")
plt.show()

### Compute Ro

In [ ]:
Ro = R_nl + alpha

## Plot params over time

In [ ]:
def add_vticks(axs, xticks, xlines=None):
    """add vertical lines to axs"""

    ## specify line style
    ax_kwargs = dict(ls="--", c="gray", lw=0.8)

    ## loop thru axs
    for ax in axs:
        ax.set_xticks(xticks)
        if xlines is not None:
            for x0 in xlines:
                ax.axvline(x0, **ax_kwargs)

    return

Plot snapshots of growth rate over time. Also, plot seasonal cycle.

In [ ]:
## specify where to evaluate R value
T0 = 2

if T0 is None:
    """for this case: average over all pos/neg values"""
    R_pos = R_nl.where(R_nl["sigma"] > 0).mean("sigma")
    R_neg = R_nl.where(R_nl["sigma"] < 0).mean("sigma")
    R_all = R_nl.mean("sigma")

else:
    """for this case: find R at given sigma value"""
    R_pos = R_nl.sel(sigma=T0, method="nearest")
    R_neg = R_nl.sel(sigma=-T0, method="nearest")
    R_all = R_nl.sel(sigma=0, method="nearest")

In [ ]:
YEARS = np.array([1871, 2011, 2051, 2081]) - 1
colors = cmocean.cm.balance(np.linspace(0.2, 0.8, len(YEARS)))

fig, axs = plt.subplots(1, 3, figsize=(8, 2.5), layout="constrained")

for c, y in zip(colors, YEARS):

    ## plot R nonlinearity
    axs[0].plot(R_nl["sigma"], R_nl_ann.sel(year=y), c=c, label=y)
    # axs[0].plot(R_nl["sigma"], R_nl.sel(year=y).isel(cycle=7), c=c, label=y)

    ## plot R seasonal cycle
    axs[1].plot(
        # params.cycle, 0.5 * (R_pos - eps).sel(year=y).squeeze(drop=True), c=c, label=y
        params.cycle,
        R_pos.sel(year=y).squeeze(drop=True),
        c=c,
        label=y,
    )
    axs[2].plot(
        # params.cycle, 0.5 * (R_neg - eps).sel(year=y).squeeze(drop=True), c=c, label=y
        params.cycle,
        R_neg.sel(year=y).squeeze(drop=True),
        c=c,
        label=y,
    )


ax_kwargs = dict(ls="--", c="k", lw=0.8)
axs[0].axvline(0, **ax_kwargs)

axs[0].set_xlabel(r"Niño 3")
axs[0].set_ylabel(r"Growth rate (yr$^{-1}$)")
axs[0].set_title(r"$R$ nonlinearity")
axs[1].set_title(r"G.R. cycle ($T>0$)")
axs[2].set_title(r"G.R. cycle ($T<0$)")
axs[0].legend()

for ax in axs:
    ax.axhline(0, **ax_kwargs)

add_vticks(axs[1:], xticks=[1, 4, 8, 12], xlines=[4, 8])
for ax in axs[1:]:
    ax.set_xlabel("Month")

src.utils.set_lims(axs[1:])
if T0 is not None:
    add_vticks(axs[:1], xticks=[-3, -T0, T0, 3], xlines=[-T0, 0, T0])
    # for tick in [-T0
    # axs[0].axvline(-.5)

plt.show()

In [ ]:
## select cycle
# SEL_CYCLE = lambda x: x.isel(cycle=slice(9, 12)).mean("cycle")
# SEL_CYCLE = lambda x: x.isel(cycle=[9,10,11]).mean("cycle")
# SEL_CYCLE = lambda x: x.isel(cycle=[6,7,8]).mean("cycle")
SEL_CYCLE = lambda x: x.mean("cycle")

fig, axs = plt.subplots(1, 3, figsize=(10, 3), layout="constrained")

## plot R and R-epsilon
for R_, c, label in zip(
    [R_pos, R_neg, R_all], ["r", "b", "k"], [r"$T>0$", r"$T<0$", "All"]
):

    ## get shared args
    plot_kwargs = dict(c=c, label=label)

    axs[0].plot(R_.year, SEL_CYCLE(R_), **plot_kwargs)
    axs[1].plot(R_.year, 0.5 * SEL_CYCLE(R_ - eps), **plot_kwargs)

## plot noise over time
delta = lambda x: x / x.isel(year=0) - 1
axs[2].plot(params.year, delta(params["xi_T"].mean("cycle")), label=r"$T$")
axs[2].plot(params.year, delta(params["xi_h"].mean("cycle")), label=r"$h$")

## label
labels = [
    r"$R$",
    r"$\frac{1}{2}\left(R-\varepsilon\right)$",
    r"$\Delta$ (stochastic forcing amp.)",
]
for ax, title in zip(axs, labels):
    ax.set_title(title)
axs[0].legend()
axs[2].legend()
axs[2].set_yticks([0, 0.1, 0.2, 0.3])
# axs[0].set_yticks([-1, 0, 1])
# axs[1].set_yticks([0, 0.6, 1.2])
# axs[2].set_yticks([-1, -1.6, -2.2])
# axs[1].set_ylim([0.5, 2.3])
# axs[2].set_ylim([-2.1, -0.3])

## formatting
axs[0].set_ylabel(r"Growth rate (yr$^{-1}$)")
axs[2].set_ylabel(r"Fraction")
ax_kwargs = dict(ls="--", c="k", lw=0.8)
add_vticks(axs, xticks=YEARS[:-1], xlines=YEARS[:-1])
for ax in axs[-1:]:
    ax.axhline(0, **ax_kwargs)

plt.show()

#### Comparison plot between $R,R_o,alpha$

In [ ]:
def plot_p(ax, p, months):
    """plot change over time for parameter"""

    ## get month indexes
    month_idxs = np.array(months) - 1

    ## helper funcs
    sel_month = lambda x: x.isel(cycle=month_idxs).mean("cycle")
    center = lambda x: x - x.isel(year=0)
    get = lambda x: center(sel_month(x))

    ## get colors for plotting
    colors = cmocean.cm.balance(np.linspace(0.2, 0.8, 4))
    colors = [colors[i] for i in range(colors.shape[0])] + ["k"]

    ## plot different T values
    for s_, color in zip([-2, -1, 1, 2, 0], colors):

        ## plot data
        ax.plot(p.year, get(p.sel(sigma=s_, method="nearest")), c=color)

    return


def plot_p_comp(axs, p0, p1, p2, months):
    """plot change over time for parameters"""

    ## plot total R, damping, and ocean contribution
    for ax, p in zip(axs, [p0, p1, p2]):

        plot_p(ax, p, months=months)

    src.utils.set_lims(axs)
    axs[0].set_yticks([-0.5, 0, 0.5])
    for ax in axs:
        ax.axhline(0, ls="--", c="k", lw=0.8)
    for ax in axs[1:]:
        ax.set_yticks([])

    src.utils.add_vticks(axs, xlines=[2010, 2060], xticks=[1870, 2010, 2060])

    return

In [ ]:
fig, axs = plt.subplots(5, 3, figsize=(8, 9), layout="constrained")

for j, (months, label) in enumerate(
    zip(
        [[1, 2, 3], [4, 5, 6], [7, 8, 9], [10, 11, 12], np.arange(1, 13)],
        ["JFM", "AMJ", "JAS", "OND", "All"],
    )
):

    plot_p_comp(axs[j], p0=R_nl, p1=-alpha, p2=Ro, months=months)
    axs[j, -1].text(s=label, x=1.1, y=0.5, transform=axs[j, -1].transAxes)

for axs_ in axs[:-1]:
    for ax in axs_:
        ax.set_xticks([])

axs[0, 0].set_title(r"$R$")
axs[0, 1].set_title(r"$-\alpha$")
axs[0, 2].set_title(r"$R_o=R+\alpha$")


# src.utils.set_lims(axs.flatten())

plt.show()

## Validate RO model

### Stochastic simulation

In [ ]:
# simulation specs
simulation_kwargs = dict(
    nyear=40,
    ncopy=1000,
    seed=1000,
    X0_ds=Th_rolling[varnames].isel(year=0, member=0, time=0),
    noise_type="white",
    use_noise_cov=False,
    is_xi_stdac=False,
)

model = src.XRO.XRO(ncycle=12, ac_order=4, is_forward=True)

In [ ]:
## do simulations
sims = get_sims_over_time(model=model, params=fits, **simulation_kwargs)

## resample to DJF
get_djf = lambda x: x.resample({"time": "QS-DEC"}).mean().isel(time=slice(4, -4, 4))

## resample to seasonal
sims_djf = get_djf(sims)
Th_djf = get_djf(Th_rolling[varnames])

### Compute stats

In [ ]:
def get_std(x):
    return x.std(["time", "member"])


def get_quantiles(x):
    return x.quantile(dim=["time", "member"], q=[0.05, 0.5, 0.95])


def get_stats(x):
    """compute relevant stats for given dataset"""

    ## empty dataset to hold results
    stats = xr.Dataset()

    ## helper func to convert to xr.DataArray
    to_da = lambda x: x.to_dataarray(dim="v")

    ## compute
    stats["sigma"] = to_da(get_std(x))
    stats["q"] = to_da(get_quantiles(x))

    return stats

In [ ]:
stats_ro = get_stats(sims_djf)
stats_gt = get_stats(Th_djf)

### Plot results

#### Quantiles

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(6, 3), layout="constrained")
for ax, stats_ in zip(axs, [stats_gt, stats_ro]):
    q = stats_["q"].sel(v=varnames[0])

    ax.plot(stats_.year, q.sel(quantile=0.95), c="r", label="El Niño")
    ax.plot(stats_.year, -q.sel(quantile=0.05), c="b", label="La Niña")
    ax.plot(
        stats_.year,
        0.5 * (q.sel(quantile=0.95) - q.sel(quantile=0.05)),
        label="Average",
        c="k",
    )

    ax_kwargs = dict(ls="--", c="gray", lw=0.8)
    add_vticks(axs, xticks=[1870, 2010, 2050], xlines=[2010, 2050])

src.utils.set_lims(axs)
axs[1].set_yticks([])
axs[1].legend()
axs[0].set_title("CESM2")
axs[1].set_title("RO")
plt.show()

#### PDFs

In [ ]:
def get_pdf(x, year, varname=varnames[0], edges=np.linspace(-4.5, 4.5, 20)):
    """compute pdf for given data"""

    ## reshape data
    x_ = x[varname].sel(year=year).stack(s=["time", "member"]).values

    ## compute
    pdf, _ = src.utils.get_empirical_pdf(x_, edges=edges)

    return pdf, edges

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(6, 2.5), layout="constrained")

for ax, x, title in zip(axs, [Th_djf, sims_djf], ["CESM2", "RO"]):

    for y, kwargs in zip(
        [1870, 2010, 2080],
        [dict(color="gray", fill=True, alpha=0.2), dict(lw=2), dict(lw=2)],
    ):

        ## compute pdf for given year
        pdf_y, edges = get_pdf(x, year=y)

        ## plot
        ax.stairs(pdf_y, edges, label=y, **kwargs)

    ## label/format
    ax.set_title(title)
    ax.axvline(0, ls="--", c="k", lw=0.8)
    ax.set_xlim([-4.3, 4.3])
    ax.set_xticks([-3, 0, 3])

## formatting
src.utils.set_lims(axs)
axs[1].set_yticks([])
axs[1].legend(prop=dict(size=10))
# axs[0].set-
plt.show()